# Quickstart: loading the geotechnical test-data archive

This notebook shows how to load the three deposited test types
(**plate sinkage**, **Bevameter shear**, **triaxial compression**) from the
**open formats only** -- `metadata.json` + the per-test `csv/*.csv` files and
the safe `*.npz` bundles. No `dill` / pickle and no custom classes are needed.

All loading goes through the shared `paperdata` helper in `../scripts/`.
Nothing here modifies any data; units, values and scientific meaning are
preserved exactly. For the interactive desktop version see
`../scripts/dashboard_qt.py`; for the web version see `../../web_dashboard`.

## 1. Imports and `paperdata` on the path

The helper lives in `../scripts/`, so we add that directory to `sys.path`.

In [ ]:
import os
import sys
import json

import numpy as np
import matplotlib.pyplot as plt

# Put the shared helper (python/scripts/paperdata.py) on the path.
SCRIPTS = os.path.abspath(os.path.join(os.getcwd(), '..', 'scripts'))
if not os.path.isdir(SCRIPTS):
    SCRIPTS = os.path.abspath(os.path.join('python', 'scripts'))
sys.path.insert(0, SCRIPTS)

import paperdata
print('Test types:', paperdata.TEST_TYPES)
print('Repo root:', paperdata.REPO_ROOT)

## 2. Load the per-test metadata (`metadata.json`)

`load_metadata` returns a list of dicts, one per test, with `test_id`,
`csv_file`, `n_rows` and type-specific scalar properties.

In [ ]:
for test_type in paperdata.TEST_TYPES:
    meta = paperdata.load_metadata(test_type)
    print(f'{test_type:14s}: {len(meta):2d} tests')

# Show the first plate-sinkage record and its available columns.
plate_meta = paperdata.load_metadata('plate_sinkage')
print('\nExample record:', plate_meta[0])
print('Array columns:', paperdata.ARRAY_COLUMNS)

## 3. Plate sinkage: pressure [kPa] vs sinkage [mm]

`load_test_csv` returns `{column_name: np.ndarray(float)}` (NaN for padding).
CSV columns are `sinkage` [mm] and `pressure` [kPa].

In [ ]:
row = paperdata.load_metadata('plate_sinkage')[0]
arrays = paperdata.load_test_csv('plate_sinkage', row['csv_file'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(arrays['sinkage'], arrays['pressure'], color='red', linewidth=1.2)
ax.set_xlabel('Sinkage [mm]')
ax.set_ylabel('Pressure [kPa]')
ax.set_title(f"Plate sinkage - {row['test_id']}")
ax.grid(True, alpha=0.3)
plt.show()

## 4. Bevameter shear: shear stress [kPa] vs displacement [mm]

CSV columns are `Displacement` [mm] and `shear_stress` [kPa].

In [ ]:
row = paperdata.load_metadata('shear')[0]
arrays = paperdata.load_test_csv('shear', row['csv_file'])
label = f"{row['Method']} | sigma_n={row['Normal_Stress']} kPa | {row['Density']} kg/m3"

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(arrays['Displacement'], arrays['shear_stress'], label=label, linewidth=1.2)
ax.set_xlabel('Displacement [mm]')
ax.set_ylabel('Shear Stress [kPa]')
ax.set_title('Bevameter shear')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.show()

## 5. Triaxial: q [kPa] vs axial strain [-]

CSV columns are `axial_strain` [-], `q` [kPa], `p_prime` [kPa] and `e` [-].
The full figure has three linked panels (q-eps_a, q-p', e-p' log-x); here we
draw the first as a minimal example.

In [ ]:
row = paperdata.load_metadata('triaxial')[0]
arrays = paperdata.load_test_csv('triaxial', row['csv_file'])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(arrays['axial_strain'], arrays['q'], color='blue', linewidth=1.2)
ax.set_xlabel('Axial Strain [-]')
ax.set_ylabel('q [kPa]')
ax.set_title(f"Triaxial - {row['test_id']}: q vs axial strain")
ax.grid(True, alpha=0.3)
plt.show()

## 6. Iterate over every test from the open formats

`iter_tests_from_csv` yields `(meta_row, arrays_dict)` for each test using only
`metadata.json` + the CSVs -- the same dill-free path used by the dashboards.

In [ ]:
for meta_row, arrays in paperdata.iter_tests_from_csv('triaxial'):
    n = len(arrays['q'])
    print(f"{meta_row['test_id']:12s}  {n:6d} rows  q_max={np.nanmax(arrays['q']):8.2f} kPa")

## 7. Safe NumPy bundle (`*.npz`, `allow_pickle=False`)

Each type also ships a single compressed `.npz` holding every test's arrays
(keys like `01_TX2_CU200__q`) plus a `_metadata_json` string. Load it with
`allow_pickle=False` -- no code executes on load, unlike the legacy `.pkl`.

In [ ]:
npz_path = os.path.join(paperdata.raw_type_dir('triaxial'), 'triaxial.npz')
z = np.load(npz_path, allow_pickle=False)

array_keys = [k for k in z.files if k != '_metadata_json']
print('First array keys:', array_keys[:4])

# The embedded metadata table is a JSON string.
meta = json.loads(str(z['_metadata_json']))
print('Tests in npz metadata:', len(meta))
print('First record:', meta[0])

# Pull one array straight out of the bundle.
key = array_keys[0]
print(f'{key}: shape={z[key].shape}, dtype={z[key].dtype}')

## Legacy `.pkl` (optional)

The original objects are still archived as `.pkl` and can be read with
`paperdata.load_pkl(test_type)` -- but that requires `dill` and the original
classes and executes code on load. Prefer the CSV / NPZ open formats above for
reuse and long-term readability.